In [1]:
import pandas as pd
import numpy as np
df = pd.read_parquet('../data/processed/features.parquet')
df.shape

(514983, 56)

In [2]:
num_features = [
    'grossapproval', 'sbaguaranteedapproval', 'terminmonths', 
    'initialinterestrate', 'jobssupported', 'bank_prior_def_rate', 
    'bank_prior_loans', 'sector_historical_default_rate', 
    'unguaranteed_exposure', 'log_gross_approval'
]

In [3]:
cat_features = [
    'naics_sector', 'businesstype', 'business_age_group', 'bank_experience_tier'
]

In [4]:
bin_features = [
    'revolverstatus', 'is_same_state_bank', 'is_variable_rate', 
    'is_franchise', 'collateralind'
]

In [5]:
all_features = num_features + cat_features + bin_features

We will use the approval year (approvalfy) to split our dataset:

Training Set: Loans approved from 2010 to 2020 (prior to 2021).
Testing Set: Loans approved from 2021 to 2026 (2021 and later).

In [6]:
train_mask = df['approvalfy'] < 2021
test_mask = df['approvalfy'] >= 2021

In [7]:
X_train = df.loc[train_mask, all_features].copy()
y_train = df.loc[train_mask, 'target'].copy()
X_test = df.loc[test_mask, all_features].copy()
y_test = df.loc[test_mask, 'target'].copy()

In [21]:
X_train.shape

(456278, 19)

In [9]:
y_train.mean()

np.float64(0.07743305616312862)

In [10]:
X_test.shape[0]

58705

In [11]:
y_test.mean()

np.float64(0.15893024444255174)

Our test set had twice the default rate of our training set because it covers the post-2021 high-inflation, rising-interest-rate environment. Testing on this period gives us a highly realistic, stress-test evaluation of how our model performs during economic downturns, rather than just in stable economic times

In [12]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [14]:
from sklearn.pipeline import Pipeline

In [15]:
num_transformer = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='median')),
    ('scaler' , StandardScaler())
])

In [16]:
cat_transformer = Pipeline(steps=[
    ('imputer' ,SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot' ,OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])    

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num',num_transformer, num_features),
        ('cat',cat_transformer, cat_features),
        ('bin','passthrough' , bin_features)
    ]
)    

In [19]:
X_train_preprocessed = preprocessor.fit_transform(X_train)

In [20]:
X_train_preprocessed.shape

(456278, 50)

***Training LogisticRegression as the baseline model before training the ensemble model***

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, classification_report

In [25]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor' , preprocessor),
    ('classifier' , LogisticRegression(max_iter=1000,random_state=42,n_jobs=-1))
])    

In [26]:
lr_pipeline.fit(X_train,y_train)

/Users/dhruvrajendrabadhe/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/dhruvrajendrabadhe/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/dhruvrajendrabadhe/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/dhruvrajendrabadhe/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/dhruvrajendrabadhe/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountere

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['grossapproval',
                                                   'sbaguaranteedapproval',
                                                   'terminmonths',
                                                   'initialinterestrate',
                                                   'jobssupported',
                                                   'bank_prior_def_rate',
                                                   'bank_prior_loans',
                                                   'sector_historical_default_rate',
                                                   'unguaranteed_exposu...
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['naics_sector',
                                                   'businesstype',
                                                   'business_age_group',
                                                   'bank_experience_tier']),
                                                 ('bin', 'passthrough',
                                                  ['revolverstatus',
                                                   'is_same_state_bank',
                                                   'is_variable_rate',
                                                   'is_franchise',
                                                   'collateralind'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, n_jobs=-1,
                                    random_state=42))])

In [27]:
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

/Users/dhruvrajendrabadhe/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/dhruvrajendrabadhe/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/dhruvrajendrabadhe/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [28]:
roc_auc = roc_auc_score(y_test, y_prob_lr)
pr_auc = average_precision_score(y_test, y_prob_lr)

In [29]:
print(roc_auc)
print(pr_auc)

0.6712826298723324
0.26424228769623737


In [30]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_lr)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[min(best_idx, len(thresholds)-1)]
best_f1 = f1_scores[best_idx]
print(f"Best Threshold for F1: {best_threshold:.4f}")
print(f"Best F1 Score: {best_f1:.4f}")

Best Threshold for F1: 0.1030
Best F1 Score: 0.3514


In [31]:
y_pred_lr = (y_prob_lr >= best_threshold).astype(int)
print("\nClassification Report at optimal threshold:")
print(classification_report(y_test, y_pred_lr, target_names=['PIF', 'CHGOFF']))


Classification Report at optimal threshold:
              precision    recall  f1-score   support

         PIF       0.90      0.61      0.73     49375
      CHGOFF       0.24      0.65      0.35      9330

    accuracy                           0.62     58705
   macro avg       0.57      0.63      0.54     58705
weighted avg       0.80      0.62      0.67     58705



### 5. Baseline Model: Logistic Regression
We trained a Logistic Regression model as our baseline. Because it is a linear model, it assumes linear relationships between features and the risk of default. 

**Results on Test Set (2021–2026):**
*   **ROC-AUC:** `0.6713` (Indicates the model has decent ranking power, significantly better than random guessing at 0.50).
*   **PR-AUC (Average Precision):** `0.2642` (Given the test set's base default rate is 15.89%, a score of 26.42% represents a clear improvement over a dummy classifier).
*   **Optimal F1 Threshold:** `0.1030` (Since defaults are the minority class, the threshold is optimized downward to maximize the F1-score).
*   **Best F1-Score:** `0.3514`

**Classification Metrics (at optimal threshold):**
*   **Recall (CHGOFF):** `65%` (The model successfully catches 65% of all actual defaults).
*   **Precision (CHGOFF):** `24%` (Out of all loans flagged as high risk, 24% actually default, while 76% are false alarms).

**Interpretation:**
The low precision is a common limitation of linear models on complex credit datasets. We expect non-linear tree-based models (like Random Forests and XGBoost) to improve these metrics by learning complex interactions between features (e.g., how the combination of a new business and a variable interest rate compounding risk).

### Training randomforest model

In [32]:
from sklearn.ensemble import RandomForestClassifier

In [33]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor' , preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100,max_depth=12,random_state=42,n_jobs=-1))
])    

In [34]:
rf_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['grossapproval',
                                                   'sbaguaranteedapproval',
                                                   'terminmonths',
                                                   'initialinterestrate',
                                                   'jobssupported',
                                                   'bank_prior_def_rate',
                                                   'bank_prior_loans',
                                                   'sector_historical_default_rate',
                                                   'unguaranteed_exposu...
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['naics_sector',
                                                   'businesstype',
                                                   'business_age_group',
                                                   'bank_experience_tier']),
                                                 ('bin', 'passthrough',
                                                  ['revolverstatus',
                                                   'is_same_state_bank',
                                                   'is_variable_rate',
                                                   'is_franchise',
                                                   'collateralind'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=12, n_jobs=-1,
                                        random_state=42))])

In [35]:
y_proba_rf = rf_pipeline.predict_proba(X_test)[:,1]

In [36]:
roc_auc_rf = roc_auc_score(y_test,y_proba_rf)
pr_auc_rf = average_precision_score(y_test,y_proba_rf)

In [37]:
print(roc_auc_rf)
print(pr_auc_rf)

0.7284305805418753
0.3348187786615606


In [39]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_rf)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold_rf = thresholds[min(best_idx, len(thresholds)-1)]
best_f1_rf = f1_scores[best_idx]
print(f"Best Threshold for F1: {best_threshold_rf:.4f}")
print(f"Best F1 Score: {best_f1_rf:.4f}")

Best Threshold for F1: 0.1192
Best F1 Score: 0.3964


In [41]:
y_pred_rf = (y_proba_rf >= best_threshold_rf).astype(int)

print(classification_report(y_test, y_pred_rf, target_names=['PIF', 'CHGOFF']))

              precision    recall  f1-score   support

         PIF       0.90      0.75      0.82     49375
      CHGOFF       0.30      0.57      0.40      9330

    accuracy                           0.72     58705
   macro avg       0.60      0.66      0.61     58705
weighted avg       0.81      0.72      0.75     58705



### 6. Ensemble Model: Random Forest
We trained a Random Forest classifier with `max_depth=12` to capture non-linear relationships and interactions.

**Results on Test Set (2021–2026):**
*   **ROC-AUC:** `0.7284` (An increase of ~5.7% over the baseline, indicating much better risk ranking).
*   **PR-AUC:** `0.3348` (A significant increase of ~7.0% over the baseline).
*   **Optimal F1 Threshold:** `0.1192`
*   **Best F1-Score:** `0.3964` (Up from 0.3514)

**Classification Metrics (at optimal threshold):**
*   **Recall (CHGOFF):** `57%`
*   **Precision (CHGOFF):** `30%` (Up from 24% - significantly reduces false alarms for loan officers).

**Interpretation:**
The ensemble of decision trees successfully captured non-linear risk factors, leading to a major performance boost. Next, we will train **XGBoost (Extreme Gradient Boosting)**, which builds trees sequentially to correct prior errors and includes parameters to directly handle class imbalance.

### Training xgboost 

In [45]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [46]:
hgb_pipeline = Pipeline(steps=[
    ('preprocessor' , preprocessor),
    ('classifier' , HistGradientBoostingClassifier(
        max_iter=300,
        max_depth=6,
        learning_rate=0.05,
        class_weight='balanced',
        random_state=42
    ))
])    

In [47]:
hgb_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['grossapproval',
                                                   'sbaguaranteedapproval',
                                                   'terminmonths',
                                                   'initialinterestrate',
                                                   'jobssupported',
                                                   'bank_prior_def_rate',
                                                   'bank_prior_loans',
                                                   'sector_historical_default_rate',
                                                   'unguaranteed_exposu...
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['naics_sector',
                                                   'businesstype',
                                                   'business_age_group',
                                                   'bank_experience_tier']),
                                                 ('bin', 'passthrough',
                                                  ['revolverstatus',
                                                   'is_same_state_bank',
                                                   'is_variable_rate',
                                                   'is_franchise',
                                                   'collateralind'])])),
                ('classifier',
                 HistGradientBoostingClassifier(class_weight='balanced',
                                                learning_rate=0.05, max_depth=6,
                                                max_iter=300,
                                                random_state=42))])

In [48]:
y_prob_hgb = hgb_pipeline.predict_proba(X_test)[:,1]

In [49]:
roc_auc_hgb = roc_auc_score(y_test, y_prob_hgb)
pr_auc_hgb = average_precision_score(y_test, y_prob_hgb)

In [50]:
print(roc_auc_hgb)
print(pr_auc_hgb)

0.7505835299225311
0.354987973882921


In [51]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_hgb)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold_hgb = thresholds[min(best_idx, len(thresholds)-1)]
best_f1_hgb = f1_scores[best_idx]
print(f"Best Threshold for F1: {best_threshold_hgb:.4f}")
print(f"Best F1 Score: {best_f1_hgb:.4f}")

Best Threshold for F1: 0.4363
Best F1 Score: 0.4175


In [52]:
y_pred_hgb = (y_prob_hgb >= best_threshold_hgb).astype(int)

print(classification_report(y_test, y_pred_hgb, target_names=['PIF', 'CHGOFF']))

              precision    recall  f1-score   support

         PIF       0.91      0.75      0.82     49375
      CHGOFF       0.32      0.61      0.42      9330

    accuracy                           0.73     58705
   macro avg       0.61      0.68      0.62     58705
weighted avg       0.82      0.73      0.76     58705



### 7. Final Model: HistGradientBoosting Classifier
We trained a Histogram-based Gradient Boosting model (scikit-learn's optimized version of LightGBM/XGBoost) with balanced class weights to address the 90/10 class imbalance.

**Results on Test Set (2021–2026):**
*   **ROC-AUC:** `0.7506` (Our best risk ranking score, a ~8% increase over Logistic Regression).
*   **PR-AUC:** `0.3550` (Our best Average Precision score, a ~9% increase over Logistic Regression).
*   **Optimal F1 Threshold:** `0.4363` (Close to 0.50 due to class weight balancing).
*   **Best F1-Score:** `0.4175`

**Classification Metrics (at optimal threshold):**
*   **Recall (CHGOFF):** `61%`
*   **Precision (CHGOFF):** `32%` (Our best precision, significantly reducing false alarms while maintaining high recall).

**Conclusion:**
HistGradientBoosting is our champion model. It successfully handles class imbalance, leverages non-linear feature splits, and runs natively and quickly on macOS. We will save this pipeline to disk for deployment in our Streamlit app and use it for SHAP explainability.

### Hyperparamter Tuning

In [53]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV

In [54]:
param_dist = {
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_iter': [100, 200, 300],
    'classifier__max_leaf_nodes': [15, 31, 63],
    'classifier__min_samples_leaf': [20, 50, 100]
}

In [55]:
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train, 
    train_size=0.1, 
    stratify=y_train, 
    random_state=42
)

In [56]:
tune_search = RandomizedSearchCV(
    hgb_pipeline,
    param_distributions=param_dist,
    n_iter=5,                    
    scoring='average_precision',  
    cv=3,                         
    random_state=42,
    n_jobs=-1
)

In [57]:
tune_search.fit(X_tune, y_tune)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['grossapproval',
                                                                                'sbaguaranteedapproval',
                                                                                'terminmonths',
                                                                                'initialinterestrate',
                                                                                'jobssupported',
                                                                                'bank_prior_def_rate',
                                                                                'bank_prior_loans',
                                                                                'sector_historical_...
                                              HistGradientBoostingClassifier(class_weight='balanced',
                                                                             learning_rate=0.05,
                                                                             max_depth=6,
                                                                             max_iter=300,
                                                                             random_state=42))]),
                   n_iter=5, n_jobs=-1,
                   param_distributions={'classifier__learning_rate': [0.01,
                                                                      0.05,
                                                                      0.1],
                                        'classifier__max_iter': [100, 200, 300],
                                        'classifier__max_leaf_nodes': [15, 31,
                                                                       63],
                                        'classifier__min_samples_leaf': [20, 50,
                                                                         100]},
                   random_state=42, scoring='average_precision')

In [58]:
tune_search.best_params_

{'classifier__min_samples_leaf': 50,
 'classifier__max_leaf_nodes': 31,
 'classifier__max_iter': 100,
 'classifier__learning_rate': 0.05}

In [59]:
tune_search.best_score_

np.float64(0.6086479284199178)

In [60]:
import joblib

In [61]:
final_hgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=100,
        max_leaf_nodes=31,
        min_samples_leaf=50,
        class_weight='balanced',
        random_state=42
    ))
])

In [62]:
final_hgb_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['grossapproval',
                                                   'sbaguaranteedapproval',
                                                   'terminmonths',
                                                   'initialinterestrate',
                                                   'jobssupported',
                                                   'bank_prior_def_rate',
                                                   'bank_prior_loans',
                                                   'sector_historical_default_rate',
                                                   'unguaranteed_exposu...
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['naics_sector',
                                                   'businesstype',
                                                   'business_age_group',
                                                   'bank_experience_tier']),
                                                 ('bin', 'passthrough',
                                                  ['revolverstatus',
                                                   'is_same_state_bank',
                                                   'is_variable_rate',
                                                   'is_franchise',
                                                   'collateralind'])])),
                ('classifier',
                 HistGradientBoostingClassifier(class_weight='balanced',
                                                learning_rate=0.05,
                                                min_samples_leaf=50,
                                                random_state=42))])

In [63]:
y_prob_final = final_hgb_pipeline.predict_proba(X_test)[:, 1]

In [64]:
roc_auc_final = roc_auc_score(y_test, y_prob_final)
pr_auc_final = average_precision_score(y_test, y_prob_final)

In [65]:
print(roc_auc_final)
print(pr_auc_final)

0.7430644666042575
0.3398705072763888


In [67]:
best_hgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        max_iter=300,       
        max_depth=6,
        learning_rate=0.05,
        class_weight='balanced',
        random_state=42
    ))
])

In [68]:
best_hgb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['grossapproval',
                                                   'sbaguaranteedapproval',
                                                   'terminmonths',
                                                   'initialinterestrate',
                                                   'jobssupported',
                                                   'bank_prior_def_rate',
                                                   'bank_prior_loans',
                                                   'sector_historical_default_rate',
                                                   'unguaranteed_exposu...
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['naics_sector',
                                                   'businesstype',
                                                   'business_age_group',
                                                   'bank_experience_tier']),
                                                 ('bin', 'passthrough',
                                                  ['revolverstatus',
                                                   'is_same_state_bank',
                                                   'is_variable_rate',
                                                   'is_franchise',
                                                   'collateralind'])])),
                ('classifier',
                 HistGradientBoostingClassifier(class_weight='balanced',
                                                learning_rate=0.05, max_depth=6,
                                                max_iter=300,
                                                random_state=42))])

In [69]:
joblib.dump(best_hgb_pipeline, '../models/best_hgb_pipeline.pkl')
print("Official champion model saved successfully!")

Official champion model saved successfully!
